In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import matplotlib as mpl
from os import path
import sys
import uproot
from tqdm import tqdm
import datetime

# local imports
# sys.path.append('../../')
sys.path.append('/exp/sbnd/app/users/munjung/xsec/cafpyana_2026Jan17/cafpyana') # absolute path for running on EAF
from analysis_village.numucc_1p0pi.categories import *
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.utils import *
from analysis_village.numucc_1p0pi.makedf.selections import *
from pyanalib.split_df_helpers import *
from pyanalib.stat_helpers import *
from pyanalib.pandas_helpers import *
from makedf.constants import *

plt.style.use("presentation.mplstyle")
cmap = mpl.cm.viridis
norm = mpl.colors.Normalize(vmin=0.0, vmax=1.0)
from matplotlib.colors import LogNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.offsetbox import AnchoredText
from matplotlib.offsetbox import AnchoredOffsetbox, DrawingArea, HPacker, VPacker, TextArea
from matplotlib.legend import Legend

mc_n_split: 1
mc_tot_pot: 1.520e+19
Integrated flux: 2.454e+11
# of targets:  1.3251484770937053e+30
xsec unit:  3.0748554553696697e-42


# Combine df chunks

In [35]:
def load_and_concat_mc_dfs(
    file_dir,
    chunk_tags=None,
    df_tag="",
    mc_keys2load=['hdr', 'evt'],
    n_max_concat=3,
    sub_dir="MC",
    sample_dir="BNB_cosmics",
):

    """
    Loops over chunk_tags (aa, ab, ...), loads mc_hdr_df and mc_evt_df, then concats them.
    Keeps the first level of multiindex value unique (__ntuple) by bumping it up by 
    the previous dfs lengths' summed.
    """

    if chunk_tags is None:
        # Determine chunk_tags automatically if not provided
        # Assume files are like 'aa.df', 'ab.df', etc. in file_dir/{sub_dir}/{sample_dir}
        import glob
        pattern = path.join(file_dir, sub_dir, sample_dir, "*.df")
        files = sorted(glob.glob(pattern))
        chunk_tags = [path.splitext(path.basename(f))[0] for f in files]

    df_lists = {k:[] for k in mc_keys2load}
    ntuple_offset = 0

    for tag in chunk_tags:
        mc_file = path.join(file_dir, sub_dir, sample_dir, tag+df_tag+".df")
        mc_split_df = pd.read_hdf(mc_file, key="split")
        mc_n_split = get_n_split(mc_file)
        print(f"Reading file with tag {tag}, mc_n_split: {mc_n_split}")
        mc_dfs = load_dfs(mc_file, mc_keys2load, n_max_concat=n_max_concat)

        # Make __ntuple unique by offsetting first level index (if exists)
        for df_key in mc_keys2load:
            df = mc_dfs[df_key]
            if isinstance(df.index, pd.MultiIndex):
                levels = list(df.index.levels)
                names = df.index.names
                # __ntuple should be at level 0
                if "__ntuple" in names:
                    idx_loc = names.index("__ntuple")
                else:
                    idx_loc = 0
                # Add offset
                new_tuples = []
                for tup in df.index:
                    tup = list(tup)
                    tup[idx_loc] = tup[idx_loc] + ntuple_offset
                    new_tuples.append(tuple(tup))
                df.index = pd.MultiIndex.from_tuples(new_tuples, names=names)
            else:
                # If single index, and it's __ntuple
                if df.index.name == "__ntuple":
                    df.index = df.index + ntuple_offset

            df_lists[df_key].append(df)

        # bump offset for next file
        if isinstance(mc_hdr_df.index, pd.MultiIndex):
            ntuple_vals = mc_hdr_df.index.get_level_values(0)
        else:
            ntuple_vals = mc_hdr_df.index
        ntuple_offset += ntuple_vals.max()+1

    concat_dfs = {k:pd.concat(df_lists[k], axis=0, sort=False) for k in df_lists.keys()}

    return concat_dfs

In [ ]:
# print(os.listdir(os.path.join(file_dir, "MC", "BNB_cosmics")))

['bj.df', 'bi_sel_mup-geniewgts.df', 'be_sel_mup-geniewgts.df', 'bf.df', 'ba_sel_mup-geniewgts.df', 'av.df', 'be.df', 'aq.df', 'at.df', 'GiBUU-sel_all-mcnu.df', 'as_sel_mup-geniewgts.df', 'az_sel_mup-geniewgts.df', 'ai_sel_mup-geniewgts.df', 'aw_sel_mup-geniewgts.df', 'ak_sel_mup-geniewgts.df', 'ar_sel_mup-geniewgts.df', 'ax_sel_mup-geniewgts.df', 'am_sel_mup-geniewgts.df', 'ag.df', 'ap_sel_mup-geniewgts.df', 'bd_sel_mup-geniewgts.df', 'ad.df', 'am.df', 'bb_sel_mup-geniewgts.df', 'aj.df', 'af_sel_mup-geniewgts.df', 'bb.df', 'GENIE_aa-sel_all-mcnu.df', 'genie_wgts-DIS', 'al_sel_mup-geniewgts.df', 'bc_sel_mup-geniewgts.df', 'au.df', 'bf_sel_mup-geniewgts.df', 'as.df', 'bc.df', 'bg.df', 'ae_sel_mup-geniewgts.df', 'bj_sel_mup-geniewgts.df', 'af.df', 'ax.df', 'al.df', 'genie_wgts-MEC', 'aj_sel_mup-geniewgts.df', 'genie_wgts-RES', 'ap.df', 'au_sel_mup-geniewgts.df', 'ab_sel_mup-geniewgts.df', 'ab.df', 'aq_sel_mup-geniewgts.df', 'ai.df', 'bh_sel_mup-geniewgts.df', 'an.df', 'ay_sel_mup-geniewg

In [ ]:
# Set your directory and tags here
file_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09"

import string
tags = []
def generate_tags(tags, end_tag="bl"):
    for first in string.ascii_lowercase:
        for second in string.ascii_lowercase:
            tag = first + second
            if tag == "bl":
                break
            tags.append(tag)
        if tag == "bl":
            break
    return tags

In [44]:
n_max_concat = 3
mc_keys2load = ['hdr', 'evt']
concat_dfs = load_and_concat_mc_dfs(
    file_dir=file_dir,
    chunk_tags=tags,
    df_tag="",
    mc_keys2load=mc_keys2load,
    n_max_concat=n_max_concat,
    sub_dir="MC",
    sample_dir="BNB_cosmics"
)

Reading file with tag aa, mc_n_split: 1
Reading file with tag ab, mc_n_split: 1
Reading file with tag ac, mc_n_split: 1
Reading file with tag ad, mc_n_split: 1
Reading file with tag ae, mc_n_split: 1
Reading file with tag af, mc_n_split: 1
Reading file with tag ag, mc_n_split: 1
Reading file with tag ah, mc_n_split: 1
Reading file with tag ai, mc_n_split: 1
Reading file with tag aj, mc_n_split: 1
Reading file with tag ak, mc_n_split: 1
Reading file with tag al, mc_n_split: 1
Reading file with tag am, mc_n_split: 1
Reading file with tag an, mc_n_split: 1
Reading file with tag ao, mc_n_split: 1
Reading file with tag ap, mc_n_split: 1
Reading file with tag aq, mc_n_split: 1
Reading file with tag ar, mc_n_split: 1
Reading file with tag as, mc_n_split: 1
Reading file with tag at, mc_n_split: 1
Reading file with tag au, mc_n_split: 1
Reading file with tag av, mc_n_split: 1
Reading file with tag aw, mc_n_split: 1
Reading file with tag ax, mc_n_split: 1
Reading file with tag ay, mc_n_split: 1


In [46]:
concat_dfs["evt"]

slc                          \
                               is_clear_cosmic      vertex               
                                                         x           y   
                                                                         
                                                                         
                                                                         
                                                                         
__ntuple  entry rec.slc..index                                           
63        0     0                            0 -163.850372  129.753403   
57        6     0                            0 -100.597557   23.539589   
16        3     0                            0  -65.908653 -151.486938   
52        6     2                            0 -115.321762  -26.646566   
55        10    0                            0  -20.671587 -179.833267   
..                                         ...         ...         ...   
186369817 18    0                            0  -93.265083   -0.754377   
186369825 3     1                            0 -166.264450   66.826683   
186369828 2     0                            0 -172.097168 -138.074615   
          7     1                            0   66.245674  -36.441566   
186369833 15    0                            0  165.951828   85.540245   

                                                                          \
                                           self    tmatch                  
                                         z            eff       pur  idx   
                                                                           
                                                                           
                                                                           
                                                                           
__ntuple  entry rec.slc..index                                             
63        0     0               125.401947   29  0.872857  0.922047  0.0   
57        6     0               288.999237   56  0.918811  0.913321  1.0   
16        3     0                70.759445   63  0.962089  0.964290  0.0   
52        6     2               296.180511   43  0.894946  0.949750  0.0   
55        10    0               307.518890   24  0.924363  0.928745  0.0   
..                                     ...  ...       ...       ...  ...   
186369817 18    0                39.073601  119  0.811516  0.847895  1.0   
186369825 3     1                78.239609   16  0.813853  0.967335  0.0   
186369828 2     0               214.908081   30  0.959033  0.963473  0.0   
          7     1               103.203369   48  0.896720  0.957244  1.0   
186369833 15    0               194.147491   28  0.921887  0.979093  2.0   

                                                       ...        mc  \
                               producer          nuid  ...        G4   
                                        crlongtrkdiry  ...  univ_190   
                                                       ...             
                                                       ...             
                                                       ...             
                                                       ...             
__ntuple  entry rec.slc..index                         ...             
63        0     0                     0     -0.545189  ...       NaN   
57        6     0                     0     -0.623331  ...       NaN   
16        3     0                     0     -0.197022  ...       NaN   
52        6     2                     0     -0.134864  ...       NaN   
55        10    0                     0      0.094636  ...       NaN   
..                                  ...           ...  ...       ...   
186369817 18    0                     0     -0.341883  ...  1.087768   
186369825 3     1                     0     -0.724106  ...  0.967026   
186369828 2     0                     0

# Get common events across detvar samples

In [ ]:
syst_dfs = {}

file_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_10"
syst_keys = ["CV", "WireMod_XThetaXW", "WireMod_YZ"]
for sidx, syst_key in enumerate(syst_keys):
    filename = "SystVar_{}_wtrks.df".format(syst_key)
    mc_file = path.join(file_dir, filename)
    mc_split_df = pd.read_hdf(mc_file, key="split")
    mc_n_split = get_n_split(mc_file)
    print("mc_n_split: %d" %(mc_n_split))
    print_keys(mc_file)

    n_max_concat = 100
    mc_keys2load = ['hdr', 'evt', 'trk'] 
    mc_dfs = load_dfs(mc_file, mc_keys2load, n_max_concat=n_max_concat)
    mc_hdr_df = mc_dfs['hdr']
    mc_evt_df = mc_dfs['evt']
    mc_trk_df = mc_dfs['trk']
    mc_trk_df = mc_trk_df[mc_trk_df.pfp.trk.producer != 4294967295]
    mask = (mc_trk_df.pfp.trk.len > 0) &\
         (mc_trk_df.pfp.pfochar.vtxdist < 100) #&\
    mc_trk_df = mc_trk_df[mask]

    nlevels = len(mc_evt_df.columns.levels)
    index_names = mc_evt_df.index.names
    mc_hdr_df.columns = pd.MultiIndex.from_tuples([tuple([str(c)] +[""] * (nlevels-1)) for c in mc_hdr_df.columns]) 
    mc_evt_df = multicol_merge(mc_evt_df.reset_index(), 
                               mc_hdr_df.reset_index(),
                               left_on=["__ntuple", "entry"],
                               right_on=["__ntuple", "entry"],
                               how="left"
                               ) 
    mc_evt_df = mc_evt_df.set_index(index_names, verify_integrity=True) 

    # need to match the nu_Es across files
    mc_evt_df["nu_E"] = mc_evt_df.mc.E

    syst_dfs[syst_key] = mc_evt_df
    syst_dfs[syst_key+"_trk"] = mc_trk_df

del mc_hdr_df
del mc_evt_df

In [ ]:
syst_keys = ["CV", "WireMod_XThetaXW", "WireMod_YZ"]

for k in syst_keys:
    syst_dfs[k] = syst_dfs[k].reset_index().set_index(["run","subrun","evt","nu_E"])
    print(len(syst_dfs[k].index))

for kidx, k in enumerate(syst_keys):
    idxs = syst_dfs[k].index
    if kidx == 0:
        common_idxs = idxs
    else:
        common_idxs = common_idxs.intersection(idxs)
print(len(common_idxs))

for k in syst_keys:
    syst_dfs[k] = syst_dfs[k].loc[common_idxs]

In [ ]:

# # file_dir = "/exp/sbnd/data/users/munjung/xsec/2025spring_v10_06_00_10"
# file_dir = "/scratch/7DayLifetime/munjung/xsec/detvars"
# syst_dfs = {}
# for sidx, syst_key in enumerate(syst_keys):
#     filename = "SystVar_{}_wtrks.df".format(syst_key)
#     mc_file = path.join(file_dir, filename)
#     mc_split_df = pd.read_hdf(mc_file, key="split")
#     mc_n_split = get_n_split(mc_file)
#     print("mc_n_split: %d" %(mc_n_split))
#     print_keys(mc_file)

#     n_max_concat = 100
#     mc_keys2load = ['hdr', 'evt', 'trk'] 
#     mc_dfs = load_dfs(mc_file, mc_keys2load, n_max_concat=n_max_concat)
#     mc_hdr_df = mc_dfs['hdr']
#     # print("total pot: %.3e" %(mc_hdr_df["pot"].sum()))
#     mc_evt_df = mc_dfs['evt']
#     mc_trk_df = mc_dfs['trk']
#     mc_trk_df = mc_trk_df[mc_trk_df.pfp.trk.producer != 4294967295]
#     mask = (mc_trk_df.pfp.trk.len > 0) &\
#          (mc_trk_df.pfp.pfochar.vtxdist < 100) #&\
#     mc_trk_df = mc_trk_df[mask]

#     nlevels = len(mc_evt_df.columns.levels)
#     index_names = mc_evt_df.index.names
#     mc_hdr_df.columns = pd.MultiIndex.from_tuples([tuple([str(c)] +[""] * (nlevels-1)) for c in mc_hdr_df.columns]) 
#     mc_evt_df = multicol_merge(mc_evt_df.reset_index(), 
#                                mc_hdr_df.reset_index(),
#                                left_on=["__ntuple", "entry"],
#                                right_on=["__ntuple", "entry"],
#                                how="left"
#                                ) 
#     mc_evt_df = mc_evt_df.set_index(index_names, verify_integrity=True) 

#     # need to match the nu_Es across files
#     mc_evt_df["nu_E"] = mc_evt_df.mc.E

#     syst_dfs[syst_key] = mc_evt_df
#     syst_dfs[syst_key+"_trk"] = mc_trk_df

# del mc_hdr_df
# del mc_evt_df

# # match common events
# for k in syst_keys:
#     syst_dfs[k] = syst_dfs[k].reset_index().set_index(["run","subrun","evt","nu_E"])
#     # print(len(syst_dfs[k].index))

# for kidx, k in enumerate(syst_keys):
#     idxs = syst_dfs[k].index
#     if kidx == 0:
#         common_idxs = idxs
#     else:
#         common_idxs = common_idxs.intersection(idxs)
# # print(len(common_idxs))

# last_level_values = common_idxs.get_level_values("nu_E")
# mask_notnan = ~pd.isna(last_level_values)
# common_idx_notnan = common_idxs[mask_notnan]

# for k in syst_keys:
#     common_nu_df = syst_dfs[k].loc[common_idx_notnan]
#     common_nu_df = common_nu_df.reset_index().set_index(["run","subrun","evt","__ntuple"])
#     common_nu_idx = common_nu_df.index
#     del common_nu_df

#     common_nu_idx = common_nu_idx.drop_duplicates()
#     common_df = syst_dfs[k].reset_index().set_index(["run","subrun","evt","__ntuple"])
#     common_df = common_df.loc[common_nu_idx]

#     this_pot = common_df[common_df["first_in_subrun"] == 1]["pot"].sum()
#     # print(this_pot)
#     if "CV" in k: 
#         cv_pot = this_pot
#         common_df["pot_weight"] = np.ones(len(common_df))
#     else:
#         pot_scale = cv_pot / this_pot
#         print(pot_scale)
#         common_df["pot_weight"] = np.ones(len(common_df)) * pot_scale

#     syst_dfs[k] = common_df

# # total POT (not used for syst sample comparison)
# # tot_pot = syst_dfs['CV'].groupby(level=[0,1]).nth(0)['pot'].sum()
# tot_pot = syst_dfs['CV'][syst_dfs['CV']['first_in_subrun'] == 1]["pot"].sum()
# print("mc_tot_pot: %.3e" %(tot_pot))
# pot_str = get_pot_str(tot_pot)


# for syst_key in syst_keys:
#     syst_dfs[syst_key] = syst_dfs[syst_key].reset_index().set_index(list(syst_dfs[f'{syst_key}_trk'].index.names)[:-1])
#     syst_dfs[syst_key+"_trk"] = get_valid_trks(syst_dfs[syst_key+"_trk"])
#     syst_dfs[syst_key+"_trk"] = match_trkdf_to_slcdf(syst_dfs[syst_key+"_trk"], syst_dfs[syst_key])

# # save matched dfs 
# save_filename = path.join(file_dir, "BNB_cosmics-matched-{}.h5".format(this_variation))
# with pd.HDFStore(save_filename, "a") as store:
#     for syst_key in syst_dfs.keys():
#         print(syst_key)
#         store.put(syst_key, syst_dfs[syst_key])
# print("save matched dfs in ", save_filename)

# Check available branches

In [1]:
import uproot

In [11]:
filename = "/pnfs/sbnd/scratch/users/munjung/ar23p/flatcafs/mc_MCP2025C_1e20_v10_06_00_09_prodgenie_corsika_proton_rockbox_sbnd_CV_caf_sbnd_zeot.caf.reweight.root"
f = uproot.open(filename)

In [12]:
wgt_names = [n for n in f["globalTree"]['global/wgts/wgts.name'].arrays(library="np")['wgts.name'][0]]

In [13]:
wgt_names

['GENIEReWeight_SBN_v1_multisigma_ZExpA1CCQE',
 'GENIEReWeight_SBN_v1_multisigma_ZExpA2CCQE',
 'GENIEReWeight_SBN_v1_multisigma_ZExpA3CCQE',
 'GENIEReWeight_SBN_v1_multisigma_ZExpA4CCQE',
 'GENIEReWeight_SBN_v1_multisigma_VecFFCCQEshape',
 'GENIEReWeight_SBN_v1_multisigma_RPA_CCQE',
 'GENIEReWeight_SBN_v1_multisigma_CoulombCCQE',
 'GENIEReWeight_SBN_v1_multisigma_NormCCMEC',
 'GENIEReWeight_SBN_v1_multisigma_NormNCMEC',
 'GENIEReWeight_SBN_v1_multisigma_DecayAngMEC',
 'GENIEReWeight_SBN_v1_multisigma_MaNCEL',
 'GENIEReWeight_SBN_v1_multisigma_EtaNCEL',
 'GENIEReWeight_SBN_v1_multisigma_MaCCRES',
 'GENIEReWeight_SBN_v1_multisigma_MvCCRES',
 'GENIEReWeight_SBN_v1_multisigma_MaNCRES',
 'GENIEReWeight_SBN_v1_multisigma_MvNCRES',
 'GENIEReWeight_SBN_v1_multisigma_NonRESBGvpCC1pi',
 'GENIEReWeight_SBN_v1_multisigma_NonRESBGvpCC2pi',
 'GENIEReWeight_SBN_v1_multisigma_NonRESBGvpNC1pi',
 'GENIEReWeight_SBN_v1_multisigma_NonRESBGvpNC2pi',
 'GENIEReWeight_SBN_v1_multisigma_NonRESBGvnCC1pi',
 'GEN

# Inspect dfs

In [25]:
import sys
sys.path.append('../../../')

from analysis_village.numucc_1p0pi.utils import *

syst_tag = "WireMod_XThetaXW_updatecalo_ccal_m"
mc_dfs = load_and_concat_mc_dfs(
    file_dir="/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_10",
    chunk_tags=[f"SystVar_{syst_tag}_wtrks"],
    df_tag="",
    keys2load=['hdr', 'evt', 'trk'],
    n_max_concat=1,
    sub_dir="",
    sample_dir=""
)

mc_hdr_df = mc_dfs['hdr']
mc_evt_df = mc_dfs['evt']
mc_trk_df = mc_dfs['trk']

Reading file with tag SystVar_WireMod_XThetaXW_updatecalo_ccal_m_wtrks, mc_n_split: 3
Keys: ['/evt_0', '/evt_1', '/evt_2', '/hdr_0', '/hdr_1', '/hdr_2', '/histgenevtdf_0', '/histgenevtdf_1', '/histgenevtdf_2', '/histpotdf_0', '/histpotdf_1', '/histpotdf_2', '/split', '/trk_0', '/trk_1', '/trk_2']


In [26]:
for c in mc_trk_df.pfp.trk.columns:
    print(c)


('producer', '', '', '')
('start', 'x', '', '')
('start', 'y', '', '')
('start', 'z', '', '')
('end', 'x', '', '')
('end', 'y', '', '')
('end', 'z', '', '')
('dir', 'x', '', '')
('dir', 'y', '', '')
('dir', 'z', '', '')
('len', '', '', '')
('rangeP', 'p_muon', '', '')
('mcsP', 'fwdP_muon', '', '')
('rangeP', 'p_pion', '', '')
('mcsP', 'fwdP_pion', '', '')
('rangeP', 'p_proton', '', '')
('mcsP', 'fwdP_proton', '', '')
('bestplane', '', '', '')
('crthit', 'distance', '', '')
('crthit', 'hit', 'time', '')
('crthit', 'hit', 'pe', '')
('chi2pid', 'I0', 'pid_ndof', '')
('chi2pid', 'I0', 'chi2_muon', '')
('chi2pid', 'I0', 'chi2_proton', '')
('chi2pid', 'I0', 'pida', '')
('chi2pid', 'I1', 'pid_ndof', '')
('chi2pid', 'I1', 'chi2_muon', '')
('chi2pid', 'I1', 'chi2_proton', '')
('chi2pid', 'I1', 'pida', '')
('chi2pid', 'I2', 'pid_ndof', '')
('chi2pid', 'I2', 'chi2_muon', '')
('chi2pid', 'I2', 'chi2_proton', '')
('chi2pid', 'I2', 'pida', '')
('truth', 'p', 'start_process', '')
('truth', 'p', 'end_